# Imports

In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path
from tqdm.auto import tqdm

# Set up paths

In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
ENRICHED_ITEMS_PATH = DATA_DIR / "Classeur1.csv" 
items_df = pd.read_csv(DATA_DIR / "items.csv")
interactions_df = pd.read_csv(DATA_DIR / "interactions_train.csv")

# Assuming you saved the merged output of the previous script here
df = pd.read_csv(ENRICHED_ITEMS_PATH, encoding='ANSI', sep=';')
df

# Ensure NaN values are treated as empty strings
# df.fillna('', inplace=True)

,item_id,genres,summary,page_count,average_rating,api_found
0,0,NaN,NaN,NaN,NaN,FAUX
1,1,Interaction analysis in education,C'est dans l'interaction en classe que s'actua...,NaN,NaN,VRAI
2,2,Narrative inquiry (Research method),Depuis la parution en 1918 de l'ouvrage fondat...,NaN,NaN,VRAI
3,3,NaN,Tome 1: Suite à un voyage humanitaire au Liban...,NaN,NaN,VRAI
4,4,French literature,"Trois histoires d'amour, un lanceur d'alerte, ...",NaN,NaN,VRAI
...,...,...,...,...,...,...
15286,15286,NaN,NaN,NaN,NaN,FAUX
15287,15287,NaN,"Jin Mori, 17 ans, est invité à un tournoi d'ar...",NaN,NaN,VRAI
15288,15288,Comics & Graphic Novels,After a disastrous defeat at the 2018 World Cu...,NaN,NaN,VRAI
15289,15289,NaN,Plusieurs années avant les événements de Red E...,NaN,NaN,VRAI


In [4]:
df.describe()

,item_id,page_count,average_rating
count,15291.000000,0.0,167.000000
mean,7645.000000,NaN,4.179641
std,4414.275818,NaN,1.044523
min,0.000000,NaN,1.000000
25%,3822.500000,NaN,4.000000
50%,7645.000000,NaN,4.500000
75%,11467.500000,NaN,5.000000
max,15290.000000,NaN,5.000000


In [12]:
1769/df.shape[0]

0.21059523809523809

In [32]:
liste = []
for i in range(df.shape[0]):
    if pd.isna(items_df["Subjects"][i]) and not df["api_found"][i]:
        liste.append(i)
len(liste)

244

In [33]:
liste

[47,
 122,
 145,
 150,
 164,
 210,
 249,
 338,
 447,
 449,
 461,
 496,
 510,
 537,
 605,
 633,
 659,
 723,
 752,
 785,
 802,
 819,
 828,
 834,
 842,
 885,
 895,
 932,
 1072,
 1073,
 1097,
 1194,
 1197,
 1198,
 1210,
 1230,
 1231,
 1402,
 1408,
 1410,
 1425,
 1430,
 1434,
 1501,
 1505,
 1520,
 1555,
 1558,
 1563,
 1643,
 1653,
 1767,
 1768,
 1795,
 1805,
 1975,
 2103,
 2105,
 2119,
 2128,
 2158,
 2162,
 2212,
 2238,
 2255,
 2349,
 2366,
 2455,
 2476,
 2517,
 2526,
 2566,
 2582,
 2583,
 2673,
 2709,
 2748,
 2760,
 2768,
 2776,
 2783,
 2897,
 2934,
 2939,
 2943,
 3005,
 3191,
 3279,
 3280,
 3320,
 3328,
 3362,
 3374,
 3383,
 3479,
 3511,
 3696,
 3702,
 3708,
 3724,
 3753,
 3810,
 3831,
 3848,
 3949,
 3986,
 4012,
 4098,
 4099,
 4101,
 4106,
 4107,
 4111,
 4115,
 4128,
 4176,
 4183,
 4208,
 4211,
 4224,
 4240,
 4242,
 4251,
 4256,
 4257,
 4270,
 4417,
 4434,
 4478,
 4518,
 4522,
 4695,
 4768,
 4813,
 4864,
 4932,
 4984,
 4992,
 4993,
 5057,
 5060,
 5083,
 5088,
 5090,
 5092,
 5094,
 5102,


In [35]:
taken = interactions_df["i"].unique().tolist()
for i in liste:
    if i not in taken:
        print(i)

210


In [36]:
items_df["Author"].value_counts()

Author
Pompéï, Christine               29
Bourdieu, Pierre                21
Ernaux, Annie                   21
Pastoureau, Michel              21
Lupano, Wilfrid                 17
                                ..
Du Maurier, Daphné               1
Fukutani, Takashi, 1952-2000     1
Park, Yong-Je                    1
Kaneshiro, Muneyuki              1
Takahiro                         1
Name: count, Length: 9357, dtype: int64

# Missing Data Strategy: Construct the 'rich_text' feature

In [ ]:
rich_texts = []

for _, row in df.iterrows():
    # Base information from your original dataset
    title = str(row.get('Title', '')).strip()
    author = str(row.get('Author', '')).strip()
    topic = str(row.get('topic', row.get('Topic', ''))).strip() 
    
    # Information from the API
    genres = str(row.get('genres', '')).strip()
    summary = str(row.get('summary', '')).strip()
    
    # Construct the string dynamically
    text_parts = []
    if title: text_parts.append(f"Title: {title}.")
    if author: text_parts.append(f"Author: {author}.")
    if topic: text_parts.append(f"Topic: {topic}.")
    
    # If the API found genres/summary, append them. Otherwise, the model 
    # just relies on the Title, Author, and Topic fallback!
    if genres: text_parts.append(f"Genres: {genres}.")
    if summary: text_parts.append(f"Summary: {summary}.")
    
    # Combine into a single string
    full_text = " ".join(text_parts)
    
    # Ultimate fallback if literally everything is missing
    if not full_text.strip():
        full_text = "Unknown book."
        
    rich_texts.append(full_text)

df['rich_text'] = rich_texts
df.to_csv(ENRICHED_ITEMS_PATH = DATA_DIR / "final_enriched_items.csv" )

# Generate Hugging Face Embeddings
## paraphrase-multilingual-MiniLM-L12-v2

In [ ]:
# all-MiniLM-L6-v2 is highly optimized, fast, and generates 384-dimensional embeddings.
# It will automatically use your RTX 4070 / 2080 Ti if PyTorch is configured for CUDA.

models = ["paraphrase-multilingual-MiniLM-L12-v2", "BAAI/bge-m3"]
output_names = ["paraphrase_multilingual", "bge_m3"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

for i in range(len(models)):
    # Generate Hugging Face Embeddings
    model = SentenceTransformer(models[i], device=device)
    print(f"Encoding text into semantic vectors (this may take a minute) using {models[i]}")
    # The encode function automatically batches the data for the GPU
    embeddings = model.encode(df['rich_text'].tolist(), show_progress_bar=True, convert_to_numpy=True)

    # Map Embeddings to Exact Item IDs
    # We need a tensor where index `i` matches the book's exact `i` ID. 
    # According to your notebook, max item ID is 15290. We create a tensor of size 15291.
    max_item_id = int(df['i'].max())
    embedding_dim = embeddings.shape[1] # Will be 384 for MiniLM

    # Initialize a tensor of zeros
    # If an item ID doesn't exist in items.csv, its embedding remains a zero vector
    item_embeddings_tensor = torch.zeros((max_item_id + 1, embedding_dim), dtype=torch.float32)

    # Populate the tensor using the specific Item IDs
    item_ids = df['i'].values.astype(int)
    item_embeddings_tensor[item_ids] = torch.tensor(embeddings, dtype=torch.float32)

    OUTPUT_TENSOR_PATH = DATA_DIR / "item_text_embeddings_"+output_names[i]+".pt"
    torch.save(item_embeddings_tensor, OUTPUT_TENSOR_PATH)
    print(f"Success! Saved embeddings tensor of shape {item_embeddings_tensor.shape} to {OUTPUT_TENSOR_PATH}")

## intfloat/multilingual-e5-base

In [ ]:
df['rich_text_e5'] = "passage: "+df["rich_text"]
df.to_csv(ENRICHED_ITEMS_PATH = DATA_DIR / "final_enriched_items.csv" )

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)

print("Encoding text into semantic vectors (this may take a minute)")
# The encode function automatically batches the data for the GPU
embeddings = model.encode(df['rich_text_e5'].tolist(), show_progress_bar=True, convert_to_numpy=True)

# We need a tensor where index `i` matches the book's exact `i` ID. 
# According to your notebook, max item ID is 15290. We create a tensor of size 15291.
max_item_id = int(df['i'].max())
embedding_dim = embeddings.shape[1] # Will be 384 for MiniLM

# If an item ID doesn't exist in items.csv, its embedding remains a zero vector
item_embeddings_tensor = torch.zeros((max_item_id + 1, embedding_dim), dtype=torch.float32)

# Populate the tensor using the specific Item IDs
item_ids = df['i'].values.astype(int)
item_embeddings_tensor[item_ids] = torch.tensor(embeddings, dtype=torch.float32)
OUTPUT_TENSOR_PATH = DATA_DIR / "item_text_embeddings_multilingual_e5.pt"
torch.save(item_embeddings_tensor, OUTPUT_TENSOR_PATH)
print(f"Success! Saved embeddings tensor of shape {item_embeddings_tensor.shape} to {OUTPUT_TENSOR_PATH}")

## EmbeddingGemma

In [ ]:
!huggingface-cli login

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# Load the Gemma embedding model directly
model = SentenceTransformer('google/embeddinggemma-300m', device=device)

print("Encoding text into semantic vectors (this may take a minute)")
embeddings = model.encode(df['rich_text'].tolist(), show_progress_bar=True, convert_to_numpy=True)
OUTPUT_TENSOR_PATH = DATA_DIR / "item_text_embeddings_multilingual_embeddinggemma.pt"
torch.save(item_embeddings_tensor, OUTPUT_TENSOR_PATH)
print(f"Success! Saved embeddings tensor of shape {item_embeddings_tensor.shape} to {OUTPUT_TENSOR_PATH}")